# HOL AI Summit - IA Multimodal con Snowflake Cortex

**Duración estimada:** 20 minutos | **Idioma:** Español

En este HOL aprenderás a procesar **imágenes, documentos y audio** con funciones de IA nativas de Snowflake, y a construir un **agente conversacional** usando lenguaje natural con **Cortex Code**.

## Qué vas a hacer
1. Analizar imágenes con AI_COMPLETE multimodal
2. Extraer datos estructurados de contratos PDF/DOCX con AI_PARSE_DOCUMENT y AI_EXTRACT
3. Transcribir y analizar sentimiento de llamadas con AI_TRANSCRIBE y AI_SENTIMENT
4. Crear un agente conversacional desde cero usando **Cortex Code en Snowsight**

> Asegúrate de tener seleccionado el warehouse `HOL_WH` y el schema `HOL_AI_SUMMIT.PUBLIC` arriba a la derecha del notebook.

## Ejercicio 1: Imágenes con IA Multimodal

Snowflake puede analizar imágenes directamente con AI_COMPLETE y modelos como Claude. Vamos a describir un siniestro vehicular y extraer datos de una cédula.

In [ ]:
-- Que imágenes tenemos en el stage?
SELECT RELATIVE_PATH, SIZE FROM DIRECTORY(@IMAGENES) ORDER BY RELATIVE_PATH;

In [ ]:
-- Analizar imagen del siniestro vehicular con IA multimodal
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Eres un perito de seguros. Describe el dano del vehiculo en esta imagen, indica severidad (leve/moderado/grave) y estima un rango de costo de reparacion en USD. Responde en español y en formato JSON con campos: descripcion, severidad, costo_estimado.',
  TO_FILE('@IMAGENES', 'choque.png')
) AS analisis_siniestro;

In [ ]:
-- Extraer datos de identificacion de una cédula (KYC)
SELECT AI_COMPLETE(
  'claude-4-sonnet',
  'Extrae los datos de esta cédula en formato JSON: numero_documento, nombres, apellidos, fecha_nacimiento, lugar_expedicion. Si no puedes leer un campo, devuelve null.',
  TO_FILE('@IMAGENES', 'cédula.jpg')
) AS datos_cédula;

## Ejercicio 2: Documentos con AI_PARSE_DOCUMENT y AI_EXTRACT

Snowflake parsea DOCX, PDF, PPTX y más formatos sin pre-procesamiento. Después podemos extraer campos estructurados con AI_EXTRACT.

In [ ]:
-- Ver los contratos ya parseados (creados por setup.sql)
SELECT file_name, LEFT(content, 300) AS preview FROM DOCS_PARSED;

In [ ]:
-- Extraer campos estructurados de los contratos de arrendamiento
SELECT
  file_name,
  AI_EXTRACT(
    content,
    ['arrendador', 'arrendatario', 'valor_canon_mensual', 'plazo_meses', 'direccion_inmueble', 'fecha_inicio']
  ) AS campos_extraidos
FROM DOCS_PARSED;

In [ ]:
-- Generar un resumen ejecutivo de cada contrato
SELECT
  file_name,
  AI_SUMMARIZE_AGG(content) AS resumen
FROM DOCS_PARSED
GROUP BY file_name;

## Ejercicio 3: Audio con AI_TRANSCRIBE y AI_SENTIMENT

Transcribimos llamadas de servicio al cliente directamente desde el stage y analizamos el sentimiento.

In [ ]:
-- Ver transcripciones (creadas por setup.sql)
SELECT file_name, sentimiento, LEFT(transcripcion, 250) AS preview FROM TRANSCRIPCIONES;

In [ ]:
-- Clasificar el motivo de la llamada con AI_CLASSIFY
SELECT
  file_name,
  sentimiento,
  AI_CLASSIFY(
    transcripcion,
    ['Reclamo', 'Consulta', 'Oferta comercial', 'Soporte tecnico', 'Otro']
  ):labels[0]::VARCHAR AS categoria
FROM TRANSCRIPCIONES;

In [ ]:
-- Análisis de sentimiento detallado + recomendaciones para el asesor
SELECT
  file_name,
  sentimiento,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Eres un coach experto en servicio al cliente. Analiza esta transcripción de una llamada y genera un JSON con los siguientes campos:\n',
      '- sentimiento_detallado: (positivo, negativo, neutro, mixto, frustrado, satisfecho)\n',
      '- nivel_urgencia: (alto, medio, bajo)\n',
      '- puntos_dolor: lista de quejas o problemas mencionados por el cliente\n',
      '- fortalezas_asesor: qué hizo bien el asesor\n',
      '- areas_mejora: qué podría mejorar el asesor\n',
      '- recomendacion_inmediata: acción que debe tomar el asesor ahora\n',
      '- script_sugerido: frase que el asesor debería usar en la próxima interacción\n',
      'Transcripción: ', transcripcion
    )
  ) AS coaching_asesor
FROM TRANSCRIPCIONES;


In [ ]:
-- Scorecard resumido del asesor por llamada
SELECT
  file_name AS llamada,
  sentimiento,
  AI_CLASSIFY(
    transcripcion,
    ['Reclamo', 'Consulta', 'Oferta comercial', 'Soporte técnico', 'Otro']
  ):labels[0]::VARCHAR AS categoría,
  AI_COMPLETE(
    'claude-4-sonnet',
    CONCAT(
      'Califica del 1 al 10 al asesor de esta llamada en: empatía, resolución, profesionalismo. ',
      'Responde SOLO un JSON con formato: {"empatia": N, "resolucion": N, "profesionalismo": N, "nota_general": N, "veredicto": "texto corto"}. ',
      'Transcripción: ', transcripcion
    )
  ) AS evaluación_asesor
FROM TRANSCRIPCIONES;


## Ejercicio 4: Cortex Code - Crea tu agente con lenguaje natural

Este es el momento estrella del HOL. Vamos a salir del notebook y usar **Cortex Code** dentro de Snowsight para construir piezas con IA escribiendo solo en español.

### Pasos
1. En Snowsight, abre **Cortex Code** (icono en el panel lateral o atajo Cmd/Ctrl+I).
2. Asegúrate de estar en el contexto: HOL_AI_SUMMIT.PUBLIC con warehouse HOL_WH.
3. Copia y pega cada uno de estos prompts (uno a la vez) y deja que Cortex Code genere el SQL/código:

**Prompt 1 - Vista de imágenes clasificadas:**
> Crea una vista llamada V_IMAGENES_CLASIFICADAS que recorra todos los archivos del stage IMAGENES y agregue una columna con la clasificación del tipo de imagen (cédula, accidente vehicular, factura, otro) usando AI_CLASSIFY sobre AI_COMPLETE multimodal.

**Prompt 2 - Tabla unificada multimodal:**
> Crea una vista llamada V_HOL_360 que combine las filas de DOCS_PARSED y TRANSCRIPCIONES en un solo dataset con columnas: tipo_fuente, archivo, contenido, sentimiento.

**Prompt 3 - Probar el agente:**
> Genera un SELECT que invoque a AI_COMPLETE con un prompt al agente AGENTE_HOL preguntándole: cuánto es el canon mensual del contrato de arrendamiento numero 2 y cuál fue el sentimiento de la última llamada?

**Prompt 4 - App de Streamlit:**
> Crea una app Streamlit in Snowflake llamada DASHBOARD_HOL que muestre: total de imágenes por tipo, sentimiento de las llamadas en gráfico de barras, y un campo de chat conectado al agente AGENTE_HOL.

Estos prompts demuestran como **una persona sin conocimientos técnicos avanzados** puede crear pipelines de IA de extremo a extremo en Snowflake en cuestión de minutos.

## Cierre - Lo que aprendiste

| Capacidad | Función / Herramienta | Diferenciador |
|-----------|----------------------|---------------|
| Análisis de imágenes | AI_COMPLETE multimodal | Sin GPU, sin APIs externas |
| Parsing de documentos | AI_PARSE_DOCUMENT | Soporta PDF, DOCX, PPTX nativo |
| Extracción estructurada | AI_EXTRACT | JSON listo para BI |
| Transcripción + sentimiento | AI_TRANSCRIBE + AI_SENTIMENT | Audio multilingüe |
| Búsqueda semántica | CORTEX SEARCH SERVICE | Sin vector DB externa |
| Agentes conversacionales | CREATE AGENT + Cortex Code | Construcción en lenguaje natural |

**Mensaje clave:** Una sola plataforma gobernada, datos sin moverse, y experiencias inteligentes construidas en lenguaje natural. Eso es Snowflake AI.